# Getting Started with OpenHydroNet (Local Dummy Run)

This notebook walks you through training and evaluating an OpenHydroNet model using a small sample dataset (5 basins) located locally in the repository.

This is designed to help you verify your installation and get a feel for the training and evaluation pipeline before running on larger datasets.

### Repository Structure Mental Model

*   **`googlehydrology/`**: The core Python package containing model architectures (under `modelzoo/`), dataset loaders (under `datasetzoo/`), and training/evaluation loops.
*   **`tutorial/`**: Contains this notebook, sample data (`Caravan-nc`), configuration templates, and helper scripts (`backend.py`).
*   **`example-configs/`**: Reference configuration files for various model setups.
*   **`run.py`**: The main entry point script (exposed as the `run` CLI command).

### Workflow Order

1.  **Configure**: Define your experiment in a YAML file (e.g., `tutorial/configs/train-config-local.yml`).
2.  **Train**: Run the training pipeline (`run train`).
3.  **Evaluate**: Run evaluation on the test period (`run evaluate`).
4.  **Analyze**: Load results and plot hydrographs (this notebook).

In [6]:
import os
import sys
import glob
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd

# Add parent directory to path to allow importing googlehydrology if needed
sys.path.append(os.path.abspath('..'))

# Import the tutorial backend helper
import backend

### Step 1: Install the package in editable mode

Before running the commands, ensure you have active the `googlehydrology` conda environment and installed the package in editable mode.
If you haven't done so, run the following command in your terminal from the root of the `flood-forecasting` repository:
```bash
pip install -e .
```
You can also run it from this notebook by uncommenting and running the cell below.

In [7]:
# !pip install -e ..

### Step 2: Run Training

We will use the pre-configured local config `configs/train-config-local.yml`.
This config is set to train for **2 epochs** on **5 basins** using CPU, which should take less than a minute.
It uses local data from `tutorial/Caravan-nc` and `Caravans_MultiMet` (which we unzipped).

Run the training using the `run` CLI command:

In [8]:
!run train --config-file configs/train-config-local.yml

/usr/local/google/home/kruparell/miniforge3/envs/googlehydrology/lib/python3.12/site-packages/dask/config.py:786: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(
[INFO] 12:57:59.114 (logging_utils.py:setup_logging) -- Logging to /usr/local/google/home/kruparell/flood-forecasting/tutorial/model-runs/5-basin-local-example_1707_125759/output.log initialized.
[INFO] 12:57:59.115 (basetrainer.py:__init__) -- ### Folder structure created at /usr/local/google/home/kruparell/flood-forecasting/tutorial/model-runs/5-basin-local-example_1707_125759
[INFO] 12:57:59.115 (basetrainer.py:__init__) -- ### Run configurations for 5-basin-local-example
[INFO] 12:57:59.115 (basetrainer.py:__init__) -- experiment_name: 5-basin-local-example
[INFO] 12:57:59.115 (basetrainer.py:__init__) -- run_dir: /usr/local/google/home/kruparell/flood-forecasting/tutorial/model-runs/5-basin-local-example_1707_125759
[INFO] 12:57:59.115 

### Step 3: Run Evaluation

After training is complete, the weights and logs are saved in a subdirectory under `model-runs/` starting with `5-basin-local-example_`.
We need to find this directory and run evaluation on the test period.

The cell below finds the latest run directory and runs `run evaluate` on it.

In [9]:
# Find the latest run directory
run_dirs = glob.glob('model-runs/5-basin-local-example_*')
if not run_dirs:
    raise FileNotFoundError("No run directories found. Did you run the training step?")

latest_run_dir = max(run_dirs, key=os.path.getmtime)
print(f"Latest run directory: {latest_run_dir}")

# Run inference on test period (saves predictions to Zarr)
!run infer --run-dir {latest_run_dir} --period test

Latest run directory: model-runs/5-basin-local-example_1707_125759
[INFO] 13:00:29.151 (logging_utils.py:setup_logging) -- Logging to model-runs/5-basin-local-example_1707_125759/output.log initialized.
[WARNING] 13:00:29.152 (warnings.py:_showwarnmsg) -- /usr/local/google/home/kruparell/miniforge3/envs/googlehydrology/lib/python3.12/site-packages/dask/config.py:786: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(

[INFO] 13:00:32.193 (tester.py:_load_weights) -- Using the model weights from model-runs/5-basin-local-example_1707_125759/model_epoch002.pt
# Inference:   0%|                                  | 0.00/8.00 [00:00<?, ?it/s]No CUDA runtime is found, using CUDA_HOME='/usr/local/google/home/kruparell/miniforge3/envs/googlehydrology'
Metrics for 1D are calculated over last 1 elements only. Ignoring 7 predictions per sequence.
# Inference: 100%|███████████████████████████| 8.00/8.00 [00:13<00:00,

### Step 4: Load and Visualize Results

Now we will load the simulation results and calculate metrics using the tutorial `backend` helper.
We will filter for the test basins defined in `basin-lists/8-basin-test.txt`.

In [10]:
# Load the test basin IDs
with open('basin-lists/8-basin-test.txt', 'r') as f:
    test_basins = set(line.strip() for line in f)

# Load data and metrics (forces recalculation for this run)
data, metrics_df = backend.load_data_and_metrics(latest_run_dir, test_basins, calculate_statistics=True)

Loading Model results from: model-runs/5-basin-local-example_1707_125759 ... simulations loaded successfully.
Calculating metrics for: Model ...

Processing Gauges:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
# Display metrics DataFrame
metrics_df.head(10)

In [ ]:
# Average metrics over all basins for each lead time
mean_metrics = metrics_df.groupby('lead_time').mean()
print("Mean metrics per lead time:")
print(mean_metrics)

In [ ]:
# Plot hydrograph for a selected basin and lead time
basin_id = 'camels_04216418' # One of the training/testing basins
lead_time = 0 # 0-day lead time (nowcast)

if basin_id in data['basin'].values:
    sim = data['streamflow_sim'].sel(basin=basin_id, lead_time=lead_time)
    obs = data['streamflow_obs'].sel(basin=basin_id)

    plt.figure(figsize=(12, 6))
    obs.plot(label='Observed', color='black', alpha=0.7)
    sim.plot(label=f'Simulated (Lead Time {lead_time}d)', color='blue', linestyle='--')
    plt.title(f'Streamflow Hydrograph for Basin {basin_id} (Lead Time {lead_time}d)')
    plt.ylabel('Discharge')
    plt.xlabel('Date')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print(f"Basin {basin_id} not found in results.")